# Phase 6 DAEAC FCBA Pretrain

This notebook pretrains the FCBA DAEAC base model on DS1 train and selects the best checkpoint on DS1 validation.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
os.environ['PYTHONUNBUFFERED'] = '1'

# Edit before running on Kaggle.
REPO_URL = 'https://github.com/YOUR_ORG/Best-thesis-in-council.git'
BRANCH = 'main'
CONFIG = 'configs/phase6_daeac_paper_fcba.yaml'

WORK = Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'
ECG = REPO / 'ecg_thesis'
print('Kaggle working:', WORK.exists())
print('Repo path:', ECG)

## 1. Clone Repo

In [ ]:
if not ECG.exists():
    assert 'YOUR_ORG' not in REPO_URL, 'Edit REPO_URL before cloning.'
    !git clone --branch {BRANCH} {REPO_URL} /kaggle/working/Best-thesis-in-council
else:
    print('Repo already exists:', ECG)
assert ECG.exists(), f'Missing repo at {ECG}. Clone or upload it first.'
os.chdir(ECG)
print(Path.cwd())
!git status --short || true

## 2. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Copy Record-Split Bundle

Attach the Kaggle dataset that contains `record_split_manifest.json`. Pretraining uses `ds1_train.npz` and `ds1_val.npz`; the validation script also checks the other configured split files.

In [ ]:
candidate_dirs = [p.parent for p in Path('/kaggle/input').glob('**/record_split_manifest.json')]
print('Candidate record-split bundle dirs:')
for p in candidate_dirs:
    print(' -', p)

assert len(candidate_dirs) == 1, f'Expected one record-split bundle, found {candidate_dirs}'
KAGGLE_DATASET_DIR = candidate_dirs[0]
DEST = Path('data/processed/phase6_daeac_record_splits')
DEST.mkdir(parents=True, exist_ok=True)

for src in sorted(KAGGLE_DATASET_DIR.glob('*.npz')):
    shutil.copy2(src, DEST / src.name)
    print('copied', src.name)

for name in ('ds1_train.npz', 'ds1_val.npz'):
    assert (DEST / name).exists(), f'Missing required DS1 file after copy: {DEST / name}'

print('Copied files:')
!ls -lh data/processed/phase6_daeac_record_splits

## 4. Validate Repo, Data, and FCBA Forward

In [ ]:
!python scripts/check_repo.py
subprocess.run([sys.executable, 'scripts/phase6_daeac_paper/00_validate_data.py', '--config', CONFIG], check=True)

## 5. Optional Smoke Pretrain

Keep this disabled for the final Kaggle run. Enable it only when checking a fresh environment.

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_paper/01_train_base.py',
        '--config', CONFIG,
        '--epochs', '1',
        '--max-source-samples', '512',
        '--max-val-samples', '512',
        '--checkpoint-prefix', 'daeac_fcba_base_smoke',
    ], check=True)

## 6. Full FCBA Pretrain

The training script saves the best checkpoint by validation macro-F1.

In [ ]:
RUN_PRETRAIN = True
if RUN_PRETRAIN:
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_paper/01_train_base.py',
        '--config', CONFIG,
    ], check=True)

## 7. Inspect Best Checkpoint

In [ ]:
summary_path = Path('outputs/phase6_daeac_paper_fcba/metrics/daeac_base_train_summary.json')
assert summary_path.exists(), f'Missing train summary: {summary_path}'
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))

best_checkpoint = Path(summary['best_checkpoint'])
assert best_checkpoint.exists(), f'Missing best checkpoint: {best_checkpoint}'
print('BEST_CHECKPOINT =', best_checkpoint)
!ls -lh outputs/phase6_daeac_paper_fcba/checkpoints

## 8. Package Outputs

In [ ]:
!zip -r /kaggle/working/phase6_daeac_fcba_pretrain_outputs.zip outputs/phase6_daeac_paper_fcba